# Node Classification with PyTorch Geometric in TopologicPy

The goal is to predict the **room type** of each node in a graph.

## What this notebook assumes

You already have a prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

## What this notebook does

1. Reads the dataset from disk
2. Verifies the CSV schema
3. Loads the dataset into `topologicpy.PyG`
4. Trains a baseline **GraphSAGE** model for **node classification**
5. Evaluates the model on validation and test nodes
6. Visualizes training history and confusion matrices
7. Exports node-level predictions


In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Imports

We use standard Python tools for file handling and tables, and `topologicpy.PyG` for the graph machine learning workflow.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.20 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Set the dataset path and baseline hyperparameters

This section is the main place to edit the notebook.

### Dataset path
Point `DATASET_PATH` to the folder that already contains the prepared clean dataset.

### Task settings
For this tutorial, the task is:

- **level** = `"node"`
- **task** = `"classification"`
- **nodeLabelType** = `"categorical"`

### Baseline model settings
These settings are intended to be a clear baseline for node classification with GraphSAGE.

In [ ]:
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification_500").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "node"          # graph | node | edge | link
TASK = "classification"           # classification | regression | link_prediction
NODE_LABEL_TYPE = "categorical"   # room type is a categorical class index

# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                     # GraphSAGE baseline
HIDDEN_DIMS = (64, 64, 64)         # 3 hidden layers -> 4-layer stack including output layer
ACTIVATION = "relu"               # practical baseline in the current PyG helper
DROPOUT = 0.0
BATCH_NORM = True
RESIDUAL = False

# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 50
OPTIMIZER = "adam"
WEIGHT_DECAY = 0.0
USE_GPU = True
RANDOM_STATE = 53
SHUFFLE = True
EARLY_STOPPING = False


## 5. Check the dataset files

Before loading the dataset into PyG, it is good practice to verify that the three required CSV files exist.

In [ ]:
required_files = [
    DATASET_PATH / "graphs.csv",
    DATASET_PATH / "nodes.csv",
    DATASET_PATH / "edges.csv",
]

for f in required_files:
    print(f"{f.name}: {'FOUND' if f.exists() else 'MISSING'} -> {f}")

if not all(f.exists() for f in required_files):
    raise FileNotFoundError(
        "One or more required CSV files are missing. "
        "Check PREPARED_DATASET_DIR before proceeding."
    )

## 6. Inspect the CSV schema

This step is pedagogically useful because it makes the node-classification setup visible.

For node classification, the important pieces are:

### `nodes.csv`
- node identifiers
- node labels
- node features
- train/validation/test masks

### `edges.csv`
- source node
- destination node
- optional edge features

### `graphs.csv`
- graph identifier
- number of nodes

In [ ]:
graphs_df = pd.read_csv(DATASET_PATH / "graphs.csv")
nodes_df = pd.read_csv(DATASET_PATH / "nodes.csv")
edges_df = pd.read_csv(DATASET_PATH / "edges.csv")

print("graphs.csv shape:", graphs_df.shape)
print("nodes.csv shape:", nodes_df.shape)
print("edges.csv shape:", edges_df.shape)

In [ ]:
print("graphs.csv columns:")
print(list(graphs_df.columns))
print()

print("nodes.csv columns:")
print(list(nodes_df.columns))
print()

print("edges.csv columns:")
print(list(edges_df.columns))

In [ ]:
feature_cols = [c for c in nodes_df.columns if c.startswith("feat_")]
mask_cols = [c for c in nodes_df.columns if c.endswith("_mask")]

summary = {
    "num_graphs": int(graphs_df["graph_id"].nunique()),
    "num_nodes": int(len(nodes_df)),
    "num_edges": int(len(edges_df)),
    "num_classes": int(nodes_df["label"].nunique()),
    "num_node_features": int(len(feature_cols)),
    "feature_columns": feature_cols,
    "mask_columns": mask_cols,
    "train_nodes": int(nodes_df["train_mask"].sum()) if "train_mask" in nodes_df.columns else None,
    "val_nodes": int(nodes_df["val_mask"].sum()) if "val_mask" in nodes_df.columns else None,
    "test_nodes": int(nodes_df["test_mask"].sum()) if "test_mask" in nodes_df.columns else None,
}

print(json.dumps(summary, indent=2))

## 7. Optional sanity checks

These checks help confirm that the prepared dataset really is ready for node classification.

In [ ]:
assert "label" in nodes_df.columns, "nodes.csv must contain a label column"
assert all(col in nodes_df.columns for col in ["train_mask", "val_mask", "test_mask"]), (
    "nodes.csv must contain train_mask, val_mask, and test_mask"
)
assert len(feature_cols) > 0, "No node feature columns found"

print("Unique labels:", sorted(nodes_df["label"].dropna().unique().tolist()))
print("Any missing labels?", bool(nodes_df["label"].isna().any()))
print("Any missing node features?", bool(nodes_df[feature_cols].isna().any().any()))

## 8. Load the dataset with TopologicPy/PyG

This is the key step where the prepared CSV files are converted into a PyG-ready dataset.

For **node classification**, the critical arguments are:

- `level="node"`
- `task="classification"`
- `nodeLabelType="categorical"`

The explicit masks in `nodes.csv` determine which nodes belong to the train, validation, and test subsets.

In [ ]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType="continuous",
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType="categorical",
)

print("Dataset loaded successfully.")
print(json.dumps(pyg.Summary(), indent=2, default=str))

## 9. Define the baseline GraphSAGE model

This notebook uses a straightforward GraphSAGE baseline.

### Why GraphSAGE?
GraphSAGE is a strong default for node classification because it combines:

- node attributes
- neighborhood information
- inductive learning over graphs

### Why these hyperparameters?
These settings are intentionally simple and clear for a tutorial:

- convolution = GraphSAGE
- hidden dimensions = `(64, 64, 64)`
- activation = `relu`
- learning rate = `0.001`
- batch size = `32`
- epochs = `50`

In [ ]:
pyg.SetHyperparameters(
        cv=CROSS_VALIDATION,
        random_state=RANDOM_STATE,
        shuffle=SHUFFLE,

        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optimizer=OPTIMIZER,
        early_stopping=EARLY_STOPPING,
        use_gpu=USE_GPU,

        conv=CONV,
        hidden_dims=HIDDEN_DIMS,
        activation=ACTIVATION,
        dropout=DROPOUT,
        batch_norm=BATCH_NORM,
        residual=RESIDUAL,
    )


print("Model and training hyperparameters have been set.")

## 10. Train the model

Training uses only the nodes where `train_mask == 1`.

The validation mask is used during training history tracking, and the test mask remains untouched until final evaluation.

In [ ]:
history = pyg.Train()

print("History keys:", list(history.keys()))
for k, v in history.items():
    if isinstance(v, (list, tuple)):
        print(f"{k}: {len(v)} values")
    else:
        print(f"{k}: {v}")

## 11. Plot the training history

This plot helps answer two common tutorial questions:

- Is the model learning?
- Is the validation curve diverging from the training curve?

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout(width=900, height=700)
fig_hist.show()

## 12. Evaluate on validation and test nodes

Validation metrics help with model checking.

Test metrics are the final held-out results and should be the main numbers reported for this tutorial run.

In [ ]:
def pretty_print_metrics(title: str, metrics: dict) -> None:
    print("
" + "=" * 80)
    print(title)
    print("=" * 80)
    df = pd.DataFrame({"metric": list(metrics.keys()), "value": list(metrics.values())})
    df = df.sort_values("metric").reset_index(drop=True)
    print(df.to_string(index=False))
    print("=" * 80)

val_metrics = pyg.Validate()
test_metrics = pyg.Test()

pretty_print_metrics("Validation metrics", val_metrics)
pretty_print_metrics("Test metrics", test_metrics)

## 13. Plot confusion matrices

Confusion matrices are especially useful in node classification because they show **which room types are confused with which other room types**.

In [ ]:
fig_val_cm = pyg.PlotConfusionMatrix(
    split="val",
    normalize=False,
    title="Validation Confusion Matrix"
)
fig_val_cm.show()

In [ ]:
fig_test_cm = pyg.PlotConfusionMatrix(
    split="test",
    normalize=False,
    title="Test Confusion Matrix"
)
fig_test_cm.show()

## 14. Predict node labels for all nodes

This step attaches predictions to the internal PyG data objects.

It is useful when you want to inspect which nodes were correctly or incorrectly classified.

In [ ]:
_ = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
print("Predictions attached to the dataset.")

## 15. Export node-level predictions to CSV

The exported table is useful for:

- error analysis
- joining predictions back to geometry
- reviewing results graph by graph

In [ ]:
def export_node_predictions(pyg_obj, output_csv: Path) -> pd.DataFrame:
    rows = []

    for graph_idx, data in enumerate(pyg_obj.data_list):
        graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else graph_idx
        y_true = data.y.detach().cpu().numpy() if hasattr(data, "y") else None
        y_pred = data.pred.detach().cpu().numpy() if hasattr(data, "pred") else None

        train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else None
        val_mask = data.val_mask.detach().cpu().numpy() if hasattr(data, "val_mask") else None
        test_mask = data.test_mask.detach().cpu().numpy() if hasattr(data, "test_mask") else None

        for node_idx in range(data.num_nodes):
            row = {
                "graph_id": graph_id,
                "node_id": node_idx,
            }
            if y_true is not None:
                row["y_true"] = int(y_true[node_idx])
            if y_pred is not None:
                row["y_pred"] = int(y_pred[node_idx])
            if train_mask is not None:
                row["train_mask"] = bool(train_mask[node_idx])
            if val_mask is not None:
                row["val_mask"] = bool(val_mask[node_idx])
            if test_mask is not None:
                row["test_mask"] = bool(test_mask[node_idx])
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df

predictions_csv = DATASET_PATH / "node_predictions_baseline.csv"
predictions_df = export_node_predictions(pyg, predictions_csv)

print(f"Saved predictions to: {predictions_csv}")
display(predictions_df.head())

## 14. Suggested tutorial extensions

Once this baseline notebook is working, sensible next steps are:

1. compare `relu` and `tanh` if your helper supports both
2. compare GraphSAGE with GCN or GIN
3. inspect class imbalance
4. inspect the worst-confused room types
5. join predictions back to room geometry for visualization

For a tutorial, however, the notebook above is a solid baseline and keeps runtime manageable.